# 🌳 LightGBM Multi-Seed Time-Series Training

Đây là hạt nhân Machine Learning đầu tiên của dự án Optiver Realized Volatility Prediction.
**Điểm đặc biệt:** 
1. Tích hợp `TimeSeriesCVSplitter` (Reverse-Engineer bằng t-SNE) để tránh rò rỉ dữ liệu.
2. Kỹ thuật **Multi-Seed Ensembling**: Với mỗi Fold, thuật toán huấn luyện 5 mô hình LightGBM với các seed ngẫu nhiên khác nhau và lấy trung bình. Việc này giảm thiểu Variance mạnh mẽ như giải pháp Hạng 1 Kaggle đã phân tích.

In [ ]:
import os
import sys
import gc
import yaml
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

from src.training.cv_splitter import TimeSeriesCVSplitter
from src.training.metrics import rmspe
from src.models.lgbm_model import LGBMModel

### 1. Nạp Cấu Hình & Dữ Liệu Features FINAL

In [ ]:
# Khởi tạo thư mục Models
MODELS_DIR = '../models/lgbm'
os.makedirs(MODELS_DIR, exist_ok=True)

# Đọc cấu hình LightGBM cơ bản (nếu chưa có sẽ sử dụng dict mặc định)
try:
    with open('../configs/lgbm_config.yaml', 'r') as f:
        lgb_config = yaml.safe_load(f)
except FileNotFoundError:
    lgb_config = {
        'objective': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'max_depth': -1,
        'num_leaves': 128,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 1,
        'n_jobs': -1
    }

print("Đang load Dataset Feather (Features_FINAL). Quá trình này sẽ diễn ra siêu tốc độ...")
df = pd.read_feather('../data/processed/features_FINAL.feather')
print(f"Đã nạp DataFrame với kích thước: {df.shape}")

### 2. Time-Series CV Recovery (Bằng t-SNE)

In [ ]:
print("Phân tích t-SNE để khôi phục ranh giới thời gian...")
cv = TimeSeriesCVSplitter(n_splits=4, random_state=42)
# Dựa vào real_price để dò tìm thời gian thực 
cv.reverse_engineer_time_order(df_train=df, df_features=df)
df['true_time_id'] = cv.recovered_time_order['true_time_id']

# Tách danh sách biến độc lập
features = [col for col in df.columns if col not in ['time_id', 'stock_id', 'target', 'true_time_id']]
print(f"Tổng số lượng Features huấn luyện: {len(features)}")

### 3. Huấn Luyện Đa Mẫu Ngẫu Nhiên (Multi-Seed Ensemble Loop)

In [ ]:
SEEDS = [0, 11, 42, 777, 2021]
oof_predictions = np.zeros(len(df))

# Quét qua 4 n_splits Fold Time-Series
for fold, (train_idx, val_idx) in enumerate(cv.split(df)):
    print(f"\n{'='*20} FOLD {fold+1} {'='*20}")
    
    X_train, y_train = df.loc[train_idx, features], df.loc[train_idx, 'target']
    X_val, y_val = df.loc[val_idx, features], df.loc[val_idx, 'target']
    
    print(f"Train len: {len(X_train)}, Val len: {len(X_val)}")
    
    # Thuật toán LightGBM hỗ trợ truyền sample weights.
    # Bí quyết để map objective RMSE sang RMSPE tự nhiên: weight = 1 / (y_true^2)
    weight_train = 1.0 / np.square(y_train)
    weight_val = 1.0 / np.square(y_val)
    
    fold_preds_seeds = []
    
    for seed in SEEDS:
        print(f"  > Đang Train Seed: {seed}...")
        # Nạp parameters và cấy seed ngẫu nhiên
        params = lgb_config.copy()
        params['seed'] = seed
        params['feature_fraction_seed'] = seed
        params['bagging_seed'] = seed
        
        model = LGBMModel(params=params)
        model.train(
            X_train=X_train, y_train=y_train,
            X_valid=X_val, y_valid=y_val,
            weight_train=weight_train,
            weight_valid=weight_val,
            num_boost_round=5000, # Vừa đủ, có Early Stopping bảo vệ
            early_stopping_rounds=300,
            verbose_eval=False # Tắt log cho gọn gàng
        )
        
        # Suy luận
        val_preds = model.predict(X_val)
        fold_preds_seeds.append(val_preds)
        
        # Lưu Model xuống ổ cứng
        model.save_model(os.path.join(MODELS_DIR, f"lgbm_fold{fold+1}_seed{seed}.pkl"))
        
    # Thực hiện Ensembling bằng cách tính trung bình 5 seed
    val_preds_avg = np.mean(fold_preds_seeds, axis=0)
    oof_predictions[val_idx] = val_preds_avg
    
    # Tính điểm OOF cho Fold này
    fold_rmspe = rmspe(y_val.values, val_preds_avg)
    print(f"👉 Điểm RMSPE OOF Fold {fold+1}: {fold_rmspe:.5f}")
    
    # Giải phóng memory
    del X_train, y_train, X_val, y_val, weight_train, weight_val, fold_preds_seeds
    gc.collect()

### 4. Kết Quả Hệ Thống

In [ ]:
# Tính toán điểm số tổng hợp. Lưu ý: Một số bản ghi ở các mốc thời gian đầu tiên 
# có thể chưa được chạy qua validation (do cơ chế KFold tịnh tiến). Chỉ tính nơi preds > 0.
valid_indexes = oof_predictions > 0
final_rmspe = rmspe(df.loc[valid_indexes, 'target'].values, oof_predictions[valid_indexes])

print("="*50)
print(f"🔥 ĐIỂM SỐ CHUNG CUỘC (FINAL OOF RMSPE) : {final_rmspe:.5f} 🔥")
print("="*50)